In [1]:
from transformers import T5Tokenizer, T5TokenizerFast, T5ForConditionalGeneration, Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import accuracy_score
import torch
from datasets import Dataset
import evaluate
import pandas as pd
import gc
import numpy as np
import json
import nltk

In [2]:
dataset_path= 'Model_dataset/synthetic_question_ans_data-v2.csv'
cv_path= "Model_dataset/cv.json"
with open(cv_path, "r") as file:
    cv_data= json.load(file)


#qa model
qa_type_model_name= 't5-small'
qa_type_model_result= '.temp/model_results/fine_tuned_question_answer_model-small'
qa_type_model= '.temp/model/fine_tuned_question_answer_model-small'



In [3]:
cv_data.keys()

dict_keys(['current_ctc', 'expected_ctc', 'personal_information', 'education', 'working_experience', 'skills', 'availability', 'others'])

### Preprocessing

In [4]:
df= pd.read_csv(dataset_path)
df= df[["question", "question_type", "answer"]]
df["answer"]= df["answer"].fillna("")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1644 entries, 0 to 1643
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       1644 non-null   object
 1   question_type  1644 non-null   object
 2   answer         1644 non-null   object
dtypes: object(3)
memory usage: 38.7+ KB


In [5]:
df["context"] = df["question_type"].map(cv_data)

In [6]:
df= df.sample(frac=1).reset_index(drop=True)
df.head()

,question,question_type,answer,context
0,What's your expected salary in INR?,expected_ctc,850000,My expected cost to company (CTC) is ₹ 850000 ...
1,What's your country of birth?,personal_information,India,Full Name: Manab Boro\nEmail Address: mboro497...
2,What's your formal training?,education,Computer Science,Current Academic Program\n Degree: 'Master ...
3,"What's your salary expectation, including perk...",expected_ctc,850000,My expected cost to company (CTC) is ₹ 850000 ...
4,What's your expected raise percentage compared...,expected_ctc,Percentage increase depending on the offer,My expected cost to company (CTC) is ₹ 850000 ...


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1644 entries, 0 to 1643
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       1644 non-null   object
 1   question_type  1644 non-null   object
 2   answer         1644 non-null   object
 3   context        1644 non-null   object
dtypes: object(4)
memory usage: 51.5+ KB


### Retraing Preparations:

In [8]:
# Preprocess data
def preprocess_data(row):
    input_text = f"question: {row['question']} context: {row['context']}"
    target_text = row['answer']
    return {"input_text": input_text, "target_text": target_text}

processed_data = df.apply(preprocess_data, axis=1)
dataset = Dataset.from_pandas(pd.DataFrame(processed_data.tolist()))

In [9]:
# Split data into train and test
train_test_split = dataset.train_test_split(test_size=0.1)
train_dataset = train_test_split["train"]
test_dataset = train_test_split["test"]

In [10]:
tokenizer = T5Tokenizer.from_pretrained(qa_type_model_name)

def tokenize_data(example):
    input_encodings = tokenizer(example["input_text"], truncation=True, padding="max_length", max_length=512)
    target_encodings = tokenizer(example["target_text"], truncation=True, padding="max_length", max_length=128)
    input_encodings["labels"] = target_encodings["input_ids"]
    return input_encodings

train_dataset = train_dataset.map(tokenize_data, batched=True)
test_dataset = test_dataset.map(tokenize_data, batched=True)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Map:   0%|          | 0/1479 [00:00<?, ? examples/s]

Map:   0%|          | 0/165 [00:00<?, ? examples/s]

In [11]:
nltk.download('punkt')

# Load metrics
rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")

def combine_compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    
    # Replace -100 in the labels as we can't decode them directly
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Preprocess for BLEU (expects list of tokens)
    bleu_preds = [pred.split() for pred in decoded_preds]
    bleu_labels = [[label.split()] for label in decoded_labels]  # BLEU expects a list of references

    # Compute ROUGE
    rouge_results = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    rouge_results = {key: value.mid.fmeasure * 100 for key, value in rouge_results.items()}
    
    # Compute BLEU
    bleu_result = bleu_metric.compute(predictions=bleu_preds, references=bleu_labels)
    bleu_score = bleu_result["bleu"] * 100  # Convert BLEU to percentage for consistency

    # Combine results
    combined_results = {
        **rouge_results,
        "bleu": bleu_score
    }
    return combined_results


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    
    # Replace -100 in the labels as we can't decode them directly
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Compute ROUGE
    rouge_results = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    rouge_results = {key: value.mid.fmeasure * 100 for key, value in rouge_results.items()}

    return rouge_results

[nltk_data] Downloading package punkt to /home/manab/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


# Train

In [12]:
model = T5ForConditionalGeneration.from_pretrained(qa_type_model_name)

In [13]:
training_args = TrainingArguments(
    output_dir=qa_type_model_result,           # Output directory
    eval_strategy="epoch",                     # Evaluate every epoch
    save_strategy="epoch",                     # Save every epoch
    learning_rate=3e-5,                        # Learning rate
    num_train_epochs=50,                       # Number of training epochs
    per_device_train_batch_size=4,             # Batch size during training
    per_device_eval_batch_size=4,              # Batch size during evaluation
    gradient_accumulation_steps=2,             # Gradient accumulation steps
    logging_dir="./logs",                      # Directory for logs
    logging_steps=10,                          # Log every 10 steps
    save_total_limit=4,                        # Limit the number of saved checkpoints
    warmup_steps=800,                          # Warmup steps for learning rate
    weight_decay=0.01,                         # Weight decay for regularization
    adam_epsilon=1e-8,                        # Epsilon for the Adam optimizer
    max_grad_norm=1.0,                        # Max gradient norm for gradient clipping
    # fp16=True,                               # Enable mixed precision training (optional)
    # use_cpu=True,                            # Force CPU usage (if necessary)
    load_best_model_at_end=True,               # Load best model at the end of training
    # metric_for_best_model="rougeL",         # Metric to monitor for best model
    # greater_is_better=True,                    # Set to True for accuracy metrics
)

In [14]:
# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    # compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)


# Clearing memory before training
torch.cuda.empty_cache()
gc.collect()

# Train the model
trainer.train()

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,8.763700,10.039437
2,0.285700,0.237941
3,0.161700,0.129286
4,0.085600,0.071236
5,0.098000,0.063722
6,0.078700,0.058157
7,0.070500,0.056375
8,0.064700,0.055575
9,0.067000,0.053379
10,0.048200,0.051943


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=3885, training_loss=0.8585350091431285, metrics={'train_runtime': 24427.1782, 'train_samples_per_second': 3.027, 'train_steps_per_second': 0.379, 'total_flos': 4203581011918848.0, 'train_loss': 0.8585350091431285, 'epoch': 21.0})

In [15]:
evaluation_results = trainer.evaluate()
evaluation_results

{'eval_loss': 0.04787278175354004,
 'eval_runtime': 39.3412,
 'eval_samples_per_second': 4.194,
 'eval_steps_per_second': 1.068,
 'epoch': 21.0}

In [16]:
trainer.save_model(qa_type_model)
tokenizer.save_pretrained(qa_type_model)

('.temp/model/fine_tuned_question_answer_model-small/tokenizer_config.json',
 '.temp/model/fine_tuned_question_answer_model-small/special_tokens_map.json',
 '.temp/model/fine_tuned_question_answer_model-small/spiece.model',
 '.temp/model/fine_tuned_question_answer_model-small/added_tokens.json')